In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Modalità di test locale di Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Usa la modalità di test locale (disponibile con `qiskit-ibm-runtime` v0.22.0 o versioni successive) per testare i programmi prima di ottimizzarli e inviarli al vero hardware quantistico. Dopo aver usato la modalità di test locale per verificare il tuo programma, l'unica modifica necessaria è il nome del backend per eseguirlo su una QPU.

Per usare la modalità di test locale, specifica uno dei fake backend da ``qiskit_ibm_runtime.fake_provider`` oppure un backend Qiskit Aer quando istanzi un primitivo Qiskit Runtime o una sessione.

- **Fake backend**: I [fake backend](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) in ``qiskit_ibm_runtime.fake_provider`` imitano il comportamento delle QPU IBM&reg; usando snapshot delle QPU. Gli snapshot delle QPU contengono informazioni importanti sulla QPU, come la coupling map, le porte di base e le proprietà dei qubit, utili per testare il transpiler ed eseguire simulazioni rumorose della QPU. Il modello di rumore dello snapshot viene applicato automaticamente durante la simulazione.
- **Simulatore Aer**: I simulatori di [Qiskit Aer](/guides/simulate-with-qiskit-aer) offrono simulazioni ad alte prestazioni in grado di gestire circuiti più grandi e [modelli di rumore personalizzati](/guides/build-noise-models). Quando usi `AerSimulator` in modalità di test locale, sono disponibili diverse opzioni per il metodo di simulazione. Consulta l'[esempio della modalità di simulazione Clifford](#clifford-sim), che mostra come simulare efficientemente circuiti Clifford con un elevato numero di qubit.

    <details>
    <summary>**Elenco dei metodi di simulazione disponibili in Qiskit Aer**</summary>

    Per ulteriori informazioni, consulta la documentazione di [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator).

    * ``"automatic"``: Metodo di simulazione predefinito. Seleziona automaticamente il metodo di simulazione in base al circuito e al modello di rumore.

    * ``"statevector"``: Una simulazione densa del vettore di stato che può campionare i risultati delle misurazioni da circuiti *ideali* con tutte le misurazioni alla fine del circuito. Per le simulazioni rumorose, ogni shot campiona casualmente un circuito rumoroso dal modello di rumore.

    * ``"density_matrix"``: Una simulazione della matrice densità che può campionare i risultati delle misurazioni da circuiti *rumorosi* con tutte le misurazioni alla fine del circuito.

    * ``"stabilizer"``: Un simulatore efficiente dello stato stabilizzatore di Clifford che può simulare circuiti Clifford rumorosi se tutti gli errori nel modello di rumore sono anch'essi errori di Clifford.

    * ``"extended_stabilizer"``: Un simulatore approssimato per circuiti Clifford + T, basato sulla decomposizione dello stato in uno stato stabilizzatore a rango. Il numero di termini cresce con il numero di porte non-Clifford (T).

    * ``"matrix_product_state"``: Un simulatore del vettore di stato basato su reti tensoriali che usa una rappresentazione Matrix Product State (MPS) per lo stato. Può essere eseguito con o senza troncamento delle dimensioni dei bond MPS, a seconda delle opzioni del simulatore. Il comportamento predefinito è senza troncamento.

    * ``"unitary"``: Una simulazione densa della matrice unitaria di un circuito ideale. Simula la matrice unitaria del circuito stesso, anziché l'evoluzione di uno stato quantistico iniziale. Questo metodo può simulare solo porte; non supporta misurazioni, reset o rumore.

    * ``"superop"``: Una simulazione densa della matrice superoperatore di un circuito ideale o rumoroso. Simula la matrice superoperatore del circuito stesso, anziché l'evoluzione di uno stato quantistico iniziale. Questo metodo può simulare porte e reset ideali e rumorosi, ma non supporta le misurazioni.

    * ``"tensor_network"``: Una simulazione basata su reti tensoriali che supporta sia il vettore di stato sia la matrice densità. Attualmente è disponibile solo per GPU ed è accelerata tramite le API `cuTensorNet` di cuQuantum.
    </details>

> **Note:** - Puoi specificare tutte le opzioni di Qiskit Runtime in modalità di test locale. Tuttavia, tutte le opzioni tranne shots vengono ignorate quando si esegue su un simulatore locale.
> - Si consiglia di installare Qiskit Aer prima di usare i fake backend o i simulatori Aer, eseguendo `pip install qiskit-aer`. I fake backend usano i simulatori Aer internamente, se disponibili, per sfruttarne le prestazioni.


## Esempio con fake backend

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Esempi con AerSimulator
Esempio con sessioni, senza rumore:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Per simulare con rumore, specifica una QPU (hardware quantistico) e inviala ad Aer. Aer costruisce un modello di rumore basato sui dati di calibrazione di quella QPU e istanzia un backend Aer con quel modello. Se preferisci, puoi [costruire un modello di rumore](/guides/build-noise-models).

> **Caution:** Una QPU può essere influenzata da diversi tipi di rumore. Il modello di rumore di Qiskit Aer usato qui ne simula solo alcuni e pertanto è probabile che sia meno severo del rumore su una QPU reale.
> 
> Per i dettagli sugli errori inclusi quando si inizializza un modello di rumore da una QPU, consulta il riferimento API Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Esempio con rumore:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Simulazione Clifford
Poiché i circuiti Clifford possono essere simulati in modo efficiente con risultati verificabili, la simulazione Clifford è uno strumento molto utile. Per un esempio approfondito, consulta [Simulazione efficiente di circuiti stabilizzatori con i primitivi Qiskit Aer](/guides/simulate-stabilizer-circuits).

Esempio:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Passi successivi
> **Tip:** - Consulta gli [esempi dettagliati sui primitivi.](/guides/primitives-examples)
>     - Leggi [Migrazione ai primitivi V2](/guides/v2-primitives).
>     - Esercitati con i primitivi seguendo la [lezione sulle funzioni di costo](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
>     - Impara come eseguire il transpiling localmente nella sezione [Transpile](/guides/transpile).
>     - Prova il tutorial [Confronta le impostazioni del transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).